In [2]:
import random
import time

import tensorflow as tf
from tensorflow.keras import Sequential

from tensorflow.keras.layers import (
    Input,
    Conv1D,
    BatchNormalization,
    Activation,
    MaxPooling1D,
    Flatten,
    Dense,
    Dropout,
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix,  precision_score, recall_score, f1_score, accuracy_score,roc_auc_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler, Normalizer, OrdinalEncoder, MinMaxScaler,  LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.feature_selection import SelectKBest, chi2, SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline

import numpy as np
import pandas as pd


In [3]:
training_data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/UNSW-NB15/UNSW_NB15_training-set.csv")
testing_data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/UNSW-NB15/UNSW_NB15_testing-set.csv")

In [4]:
darpa_pcap = pd.read_csv("/content/drive/MyDrive/Array_submission/Dataset/UNSW-features/Darpa/Darpa_Pcap_UNSW_features.csv")
unsw_pcap = pd.read_csv("/content/drive/MyDrive/Array_submission/Dataset/UNSW-features/UNSW-NB15/UNSW_Pcap_UNSW_features.csv")
cic2017_pcap = pd.read_csv("/content/drive/MyDrive/Array_submission/Dataset/UNSW-features/CIC2017/CIC_Pcap_UNSW_features.csv")
cic2018_pcap = pd.read_csv("/content/drive/MyDrive/Array_submission/Dataset/UNSW-features/CIC2018/CIC18_Pcap_UNSW_features.csv")


In [5]:
training_data.columns

Index(['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes',
       'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss',
       'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin',
       'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth',
       'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm',
       'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm',
       'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm',
       'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label'],
      dtype='object')

In [6]:
darpa_pcap.columns

Index(['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes',
       'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'sload', 'dload',
       'spkts', 'dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz',
       'dmeansz', 'trans_depth', 'res_bdy_len', 'sjit', 'djit', 'stime',
       'ltime', 'sintpkt', 'dintpkt', 'tcprtt', 'synack', 'ackdat',
       'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login',
       'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm',
       'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'label',
       'attack_cat'],
      dtype='object')

In [7]:
unsw_pcap.columns

Index(['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes',
       'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'sload', 'dload',
       'spkts', 'dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz',
       'dmeansz', 'trans_depth', 'res_bdy_len', 'sjit', 'djit', 'stime',
       'ltime', 'sintpkt', 'dintpkt', 'tcprtt', 'synack', 'ackdat',
       'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login',
       'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm',
       'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'label',
       'category'],
      dtype='object')

In [8]:
cic2017_pcap.columns

Index(['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes',
       'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'sload', 'dload',
       'spkts', 'dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz',
       'dmeansz', 'trans_depth', 'res_bdy_len', 'sjit', 'djit', 'stime',
       'ltime', 'sintpkt', 'dintpkt', 'tcprtt', 'synack', 'ackdat',
       'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login',
       'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm',
       'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'label',
       'category'],
      dtype='object')

In [9]:
cic2018_pcap.columns

Index(['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes',
       'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'sload', 'dload',
       'spkts', 'dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz',
       'dmeansz', 'trans_depth', 'res_bdy_len', 'sjit', 'djit', 'stime',
       'ltime', 'sintpkt', 'dintpkt', 'tcprtt', 'synack', 'ackdat',
       'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login',
       'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm',
       'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'Label',
       'source_file'],
      dtype='object')

In [10]:
cic2017_pcap.drop(['label'], axis=1, inplace=True)
cic2017_pcap.rename(columns={"category": "label"}, inplace=True)
unsw_pcap.drop(['label'], axis=1, inplace=True)
unsw_pcap.rename(columns={"category": "label"}, inplace=True)
cic2018_pcap.rename(columns={"Label": "label"}, inplace=True)
darpa_pcap.drop(['label'], axis=1, inplace=True)
darpa_pcap.rename(columns={"attack_cat": "label"}, inplace=True)

In [11]:
darpa_pcap['source_pcap'] = "darpa"
unsw_pcap['source_pcap'] = "unsw-nb15"
cic2017_pcap['source_pcap'] = "cic2017"
cic2018_pcap['source_pcap'] = "cic2018"

In [12]:
training_data.drop(['label'], axis=1, inplace=True)
testing_data.drop(['label'], axis=1, inplace=True)

In [13]:
rename_dict = {
    'smean': 'smeansz',
    'dmean': 'dmeansz',
    'response_body_len': 'res_bdy_len',
    'sinpkt': 'sintpkt',
    'dinpkt': 'dintpkt',
    'attack_cat': 'label'
}

training_data = training_data.rename(columns=rename_dict)
testing_data = testing_data.rename(columns=rename_dict)

common_cols_in_df2_order = [col for col in darpa_pcap.columns if col in training_data.columns]

training_data = training_data[common_cols_in_df2_order]
testing_data = testing_data[common_cols_in_df2_order]

In [14]:
darpa_pcap.columns

Index(['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes',
       'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'sload', 'dload',
       'spkts', 'dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz',
       'dmeansz', 'trans_depth', 'res_bdy_len', 'sjit', 'djit', 'stime',
       'ltime', 'sintpkt', 'dintpkt', 'tcprtt', 'synack', 'ackdat',
       'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login',
       'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm',
       'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'label',
       'source_pcap'],
      dtype='object')

In [15]:
data_list = [darpa_pcap,unsw_pcap,cic2017_pcap,cic2018_pcap]
df = pd.concat(data_list)
df.shape

(5997641, 50)

In [16]:
df["label"].value_counts()

,count
label,
Normal,3107871
BENIGN,2271658
Infiltration,174764
DoS Hulk,163790
dos,78264
DoS-GoldenEye,68127
DoS-Slowloris,35252
DoS GoldenEye,24167
DoS Slowhttptest,20104


In [17]:
df['source_pcap'].value_counts()

,count
source_pcap,
darpa,2880337
cic2017,2443090
cic2018,337880
unsw-nb15,336334


In [ ]:
import re
import pandas as pd

LABEL_COLUMN = "label"

df["original_label"] = df[LABEL_COLUMN].copy()

labels = (
    df[LABEL_COLUMN]
    .astype("string")
    .str.strip()
    .str.replace(
        r"(?:\s*[-_]\s*|\s+)attempted\s*$",
        "",
        regex=True,
        flags=re.IGNORECASE,
    )
    .str.strip()
)

normalized = (
    labels
    .str.casefold()
    .str.replace(r"[_-]+", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df[LABEL_COLUMN] = labels

benign_mask = normalized.isin(["normal", "benign"])
df.loc[benign_mask, LABEL_COLUMN] = "benign"

recon_mask = normalized.isin(["probe", "reconnaissance"])
df.loc[recon_mask, LABEL_COLUMN] = "reconnaissance"

dos_mask = normalized.str.match(r"^dos(?:\s|$)", na=False)
df.loc[dos_mask, LABEL_COLUMN] = "dos"

backdoor_mask = normalized.isin(["backdoor", "backdoors"])
df.loc[backdoor_mask, LABEL_COLUMN] = "backdoor"

df.loc[normalized.eq("r2l"), LABEL_COLUMN] = "r2l"
df.loc[normalized.eq("u2r"), LABEL_COLUMN] = "u2r"

df[LABEL_COLUMN] = df[LABEL_COLUMN].str.casefold()

display(
    df[LABEL_COLUMN]
    .value_counts(dropna=False)
    .to_frame("count")
)

,count
label,
benign,5379529
dos,408805
infiltration,174764
reconnaissance,15247
exploits,5840
fuzzers,4133
web attack - brute force,3084
r2l,2148
web attack - xss,1561


In [ ]:
source_label_counts = (
    df.groupby(["source_pcap", "label"], dropna=False)
      .size()
      .reset_index(name="count")
)

source_totals = (
    source_label_counts.groupby("source_pcap")["count"]
    .transform("sum")
)

source_label_counts["within_source_pct"] = (
    100 * source_label_counts["count"] / source_totals
)

display(
    source_label_counts.sort_values(
        ["source_pcap", "count"],
        ascending=[True, False]
    )
)

,source_pcap,label,count,within_source_pct
0,cic2017,benign,2211658,90.527078
1,cic2017,dos,226254,9.260977
4,cic2017,web attack - brute force,3084,0.126234
6,cic2017,web attack - xss,1561,0.063894
3,cic2017,infiltration,263,0.010765
2,cic2017,heartbleed,246,0.010069
5,cic2017,web attack - sql injection,24,0.000982
9,cic2018,infiltration,174501,51.645851
8,cic2018,dos,103379,30.596366
7,cic2018,benign,60000,17.757784


In [20]:
df.columns

Index(['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes',
       'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'sload', 'dload',
       'spkts', 'dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz',
       'dmeansz', 'trans_depth', 'res_bdy_len', 'sjit', 'djit', 'stime',
       'ltime', 'sintpkt', 'dintpkt', 'tcprtt', 'synack', 'ackdat',
       'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login',
       'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm',
       'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'label',
       'source_pcap', 'source_file', 'original_label'],
      dtype='object')

In [21]:
protocol_counts = (
    df.groupby(["source_pcap", "proto"], dropna=False)
      .size()
      .reset_index(name="count")
)

protocol_totals = (
    protocol_counts.groupby("source_pcap")["count"]
    .transform("sum")
)

protocol_counts["within_source_pct"] = (
    100 * protocol_counts["count"] / protocol_totals
)

display(
    protocol_counts.sort_values(
        ["source_pcap", "count"],
        ascending=[True, False]
    )
)

,source_pcap,proto,count,within_source_pct
6,cic2017,tcp,1738401,71.155831
7,cic2017,udp,650344,26.619732
0,cic2017,2054,31306,1.281410
2,cic2017,hopopt,21447,0.877864
3,cic2017,icmp,911,0.037289
...,...,...,...,...
267,unsw-nb15,esp,2,0.000595
269,unsw-nb15,hopopt,2,0.000595
272,unsw-nb15,ipv4,2,0.000595
124,unsw-nb15,193,1,0.000297


In [22]:
source_label_protocol = (
    df.groupby(
        ["source_pcap", "label", "proto"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
)

display(
    source_label_protocol.sort_values(
        ["source_pcap", "label", "count"],
        ascending=[True, True, False]
    )
)

,source_pcap,label,proto,count
6,cic2017,benign,tcp,1506970
7,cic2017,benign,udp,650344
0,cic2017,benign,2054,31306
2,cic2017,benign,hopopt,21447
3,cic2017,benign,icmp,910
...,...,...,...,...
306,unsw-nb15,generic,tcp,722
307,unsw-nb15,generic,udp,14
308,unsw-nb15,reconnaissance,tcp,1538
309,unsw-nb15,shellcode,tcp,147


In [ ]:
source_share = (
    df["source_pcap"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("overall_share_pct")
    .reset_index()
    .rename(columns={"index": "source_pcap"})
)

attack_df = df[df["label"] != "benign"]

attack_source_share = (
    attack_df["source_pcap"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("attack_share_pct")
    .reset_index()
    .rename(columns={"index": "source_pcap"})
)

display(source_share)
display(attack_source_share)

,source_pcap,overall_share_pct
0,darpa,48.024498
1,cic2017,40.734182
2,cic2018,5.633548
3,unsw-nb15,5.607771


,source_pcap,attack_share_pct
0,cic2018,44.956254
1,cic2017,37.441758
2,darpa,15.425683
3,unsw-nb15,2.176305


In [24]:

df_master = df.copy()
df_master["feature_schema"] = "unsw_nb15"
df_master["binary_label"] = np.where(
    df_master["label"].eq("benign"),
    "benign",
    "attack",
)

MATCHED_CATEGORY_SOURCES = {
    "dos": {"darpa", "unsw-nb15", "cic2017", "cic2018"},
    "reconnaissance": {"darpa", "unsw-nb15"},
}


def assign_category_scope(row):
    if row["label"] == "benign":
        return "benign"
    valid_sources = MATCHED_CATEGORY_SOURCES.get(row["label"])
    if valid_sources is not None and row["source_pcap"] in valid_sources:
        return "matched_category"
    return "source_specific_or_unmatched"


df_master["category_scope"] = df_master.apply(
    assign_category_scope,
    axis=1,
)

print(df_master.shape)
display(
    df_master.groupby(
        ["source_pcap", "label", "binary_label", "category_scope"],
        dropna=False,
    ).size().reset_index(name="count")
)


(5997641, 54)


,source_pcap,label,binary_label,category_scope,count
0,cic2017,benign,benign,benign,2211658
1,cic2017,dos,attack,matched_category,226254
2,cic2017,heartbleed,attack,source_specific_or_unmatched,246
3,cic2017,infiltration,attack,source_specific_or_unmatched,263
4,cic2017,web attack - brute force,attack,source_specific_or_unmatched,3084
5,cic2017,web attack - sql injection,attack,source_specific_or_unmatched,24
6,cic2017,web attack - xss,attack,source_specific_or_unmatched,1561
7,cic2018,benign,benign,benign,60000
8,cic2018,dos,attack,matched_category,103379
9,cic2018,infiltration,attack,source_specific_or_unmatched,174501


In [ ]:
from pathlib import Path
import gc
import json
import random
import time
import warnings

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)

SOURCE_COLUMN = "source_pcap"
LABEL_COLUMN = "label"
BINARY_COLUMN = "binary_label"
RANDOM_STATE = 42
EXTERNAL_TEST_SIZE = 0.30
GROUP_COLUMN = None

OUTPUT_DIR = Path("unsw_nb15_prepared_experiments")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = Path("unsw_nb15_model_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [11, 22, 33, 44, 55]


SOURCE_ROLE = {
    "unsw-nb15": "same_domain_reextraction",
    "darpa": "heterogeneous",
    "cic2017": "heterogeneous",
    "cic2018": "heterogeneous",
}

SHARED_CATEGORIES_BY_SOURCE = {
    "darpa": {"benign", "dos", "reconnaissance"},
    "unsw-nb15": {"benign", "dos", "reconnaissance"},
    "cic2017": {"benign", "dos"},
    "cic2018": {"benign", "dos"},
}


In [ ]:
UNSW_FEATURE_COLUMNS = [
     "proto", "state", "dur","sbytes", "dbytes", "sttl", "dttl", "sloss", "dloss", "service",
    "sload", "dload", "spkts", "dpkts", "swin", "dwin", "stcpb",
    "dtcpb", "smeansz", "dmeansz", "trans_depth", "res_bdy_len",
    "sjit", "djit", "sintpkt", "dintpkt", "tcprtt",
    "synack", "ackdat", "is_sm_ips_ports", "ct_state_ttl",
    "ct_flw_http_mthd", "is_ftp_login", "ct_ftp_cmd", "ct_srv_src",
    "ct_srv_dst", "ct_dst_ltm", "ct_src_ltm", "ct_src_dport_ltm",
    "ct_dst_sport_ltm", "ct_dst_src_ltm",
]


def normalize_attack_category(value):
    if pd.isna(value):
        return pd.NA
    value = str(value).strip().casefold()
    value = value.replace("- attempted", "").strip()
    if value in {"normal", "benign"}:
        return "benign"
    if value.startswith("dos"):
        return "dos"
    if value in {"probe", "reconnaissance"}:
        return "reconnaissance"
    if value in {"backdoor", "backdoors"}:
        return "backdoor"
    return value


def normalize_native_unsw(frame, split_name):
    result = frame.copy()

    if "label" in result.columns:
        numeric = pd.to_numeric(result["label"], errors="coerce")
        if numeric.notna().all() and set(numeric.unique()).issubset({0, 1}):
            result["official_binary_label"] = numeric.astype("int8")

    if "attack_cat" in result.columns:
        result["original_label"] = result["attack_cat"]
        result[LABEL_COLUMN] = result["attack_cat"].map(normalize_attack_category)
    elif "label" in result.columns and not pd.api.types.is_numeric_dtype(result["label"]):
        result["original_label"] = result["label"]
        result[LABEL_COLUMN] = result["label"].map(normalize_attack_category)
    else:
        raise KeyError(
            "Native UNSW data require attack_cat, or a textual label column."
        )

    result[BINARY_COLUMN] = np.where(
        result[LABEL_COLUMN].eq("benign"),
        "benign",
        "attack",
    )
    result[SOURCE_COLUMN] = "unsw-native"
    result["native_split"] = split_name
    result["feature_schema"] = "unsw_nb15"

    result = result.drop(columns=["id", "attack_cat"], errors="ignore")
    return result


original_train = normalize_native_unsw(training_data, "train")
original_test = normalize_native_unsw(testing_data, "test")

missing_train = sorted(set(UNSW_FEATURE_COLUMNS) - set(original_train.columns))
missing_test = sorted(set(UNSW_FEATURE_COLUMNS) - set(original_test.columns))
if missing_train or missing_test:
    raise KeyError(
        f"Native feature mismatch. Train missing={missing_train}; "
        f"test missing={missing_test}"
    )

print("Original train:", original_train.shape)
print("Original test:", original_test.shape)
display(original_train[LABEL_COLUMN].value_counts(dropna=False))
display(original_test[LABEL_COLUMN].value_counts(dropna=False))



_native_compare_columns = UNSW_FEATURE_COLUMNS + [LABEL_COLUMN]
_native_train_hash = pd.util.hash_pandas_object(
    original_train[_native_compare_columns],
    index=False,
).value_counts().sort_index()
_native_test_hash = pd.util.hash_pandas_object(
    original_test[_native_compare_columns],
    index=False,
).value_counts().sort_index()

if (
    len(original_train) == len(original_test)
    and _native_train_hash.equals(_native_test_hash)
):
    raise RuntimeError(
        "Native UNSW training and testing sets are identical. "
        "Check the CSV paths and the column-alignment cell before running "
        "Experiment 4; training on the test set would invalidate the results."
    )

native_attack_pct = 100.0 * (
    original_test[BINARY_COLUMN].eq("attack").mean()
)
print(f"Native UNSW test attack prevalence: {native_attack_pct:.4f}%")


Original train: (175341, 47)
Original test: (82332, 47)


,count
label,
benign,56000
generic,40000
exploits,33393
fuzzers,18184
dos,12264
reconnaissance,10491
analysis,2000
backdoor,1746
shellcode,1133


,count
label,
benign,37000
generic,18871
exploits,11132
fuzzers,6062
dos,4089
reconnaissance,3496
analysis,677
backdoor,583
shellcode,378


Native UNSW test attack prevalence: 55.0600%


In [ ]:

def source_label_distribution(frame, label_column=LABEL_COLUMN):
    table = (
        frame.groupby([SOURCE_COLUMN, label_column], dropna=False)
        .size().reset_index(name="count")
    )
    table["within_source_pct"] = (
        100 * table["count"]
        / table.groupby(SOURCE_COLUMN)["count"].transform("sum")
    )
    return table.sort_values(
        [SOURCE_COLUMN, "count"],
        ascending=[True, False],
    )


def split_each_source(frame, stratify_column, random_state=RANDOM_STATE):
    train_parts, test_parts = [], []

    for source, source_frame in frame.groupby(SOURCE_COLUMN, sort=True):
        source_frame = source_frame.copy()

        if GROUP_COLUMN is not None and GROUP_COLUMN in source_frame.columns:
            splitter = GroupShuffleSplit(
                n_splits=1,
                test_size=EXTERNAL_TEST_SIZE,
                random_state=random_state,
            )
            train_idx, test_idx = next(
                splitter.split(
                    source_frame,
                    groups=source_frame[GROUP_COLUMN],
                )
            )
            train_part = source_frame.iloc[train_idx].copy()
            test_part = source_frame.iloc[test_idx].copy()
        else:
            counts = source_frame[stratify_column].value_counts(dropna=False)
            stratify = source_frame[stratify_column] if counts.min() >= 2 else None
            if stratify is None:
                warnings.warn(
                    f"{source}: unstratified split because a class has <2 rows."
                )
            train_part, test_part = train_test_split(
                source_frame,
                test_size=EXTERNAL_TEST_SIZE,
                random_state=random_state,
                stratify=stratify,
            )

        train_part["split"] = "train"
        test_part["split"] = "test"
        train_parts.append(train_part)
        test_parts.append(test_part)

    return (
        pd.concat(train_parts, ignore_index=True),
        pd.concat(test_parts, ignore_index=True),
    )


def balance_shared_external_training(frame, random_state=RANDOM_STATE):
    selected_attacks, allocation = [], []

    for category in ["dos", "reconnaissance"]:
        category_frame = frame[frame[LABEL_COLUMN].eq(category)]
        counts = category_frame.groupby(SOURCE_COLUMN).size()
        if len(counts) < 2:
            warnings.warn(f"Skipping {category}: present in fewer than two sources.")
            continue
        quota = int(counts.min())
        for source in sorted(counts.index):
            cell = category_frame[category_frame[SOURCE_COLUMN].eq(source)]
            selected_attacks.append(
                cell.sample(n=quota, replace=False, random_state=random_state)
            )
            allocation.append({
                "label": category,
                "source_pcap": source,
                "available": len(cell),
                "selected": quota,
            })

    if not selected_attacks:
        raise ValueError("No shared attack categories available for balancing.")

    attacks = pd.concat(selected_attacks, ignore_index=True)
    benign_parts = []
    for source, attack_count in attacks.groupby(SOURCE_COLUMN).size().items():
        benign = frame[
            frame[SOURCE_COLUMN].eq(source)
            & frame[LABEL_COLUMN].eq("benign")
        ]
        if len(benign) < attack_count:
            raise ValueError(f"{source}: insufficient benign rows for balancing.")
        benign_parts.append(
            benign.sample(
                n=int(attack_count),
                replace=False,
                random_state=random_state,
            )
        )

    balanced = pd.concat([attacks, *benign_parts], ignore_index=True)
    balanced = balanced.sample(frac=1, random_state=random_state).reset_index(drop=True)
    return balanced, pd.DataFrame(allocation)


def balance_binary_external_training(frame, random_state=RANDOM_STATE):
    counts = frame.groupby([SOURCE_COLUMN, BINARY_COLUMN]).size().unstack(fill_value=0)
    if not {"benign", "attack"}.issubset(counts.columns):
        raise ValueError(f"Both classes are required per source:\n{counts}")
    quota = int(counts[["benign", "attack"]].min().min())
    parts = []
    for source in sorted(frame[SOURCE_COLUMN].unique()):
        for target in ["benign", "attack"]:
            cell = frame[
                frame[SOURCE_COLUMN].eq(source)
                & frame[BINARY_COLUMN].eq(target)
            ]
            parts.append(
                cell.sample(n=quota, replace=False, random_state=random_state)
            )
    balanced = pd.concat(parts, ignore_index=True)
    return balanced.sample(frac=1, random_state=random_state).reset_index(drop=True), quota


def save_frame(frame, filename):
    path = OUTPUT_DIR / filename
    frame.to_csv(path, index=False)
    print(f"Saved {len(frame):,} rows -> {path}")
    return path



TARGET_TEST_ATTACK_FRACTION = 0.551
TARGET_TEST_BENIGN_FRACTION = 1.0 - TARGET_TEST_ATTACK_FRACTION


def _largest_remainder_allocation(counts, requested_total):
    """Allocate a sample across strata while preserving proportions."""
    counts = pd.Series(counts, dtype="int64")
    available_total = int(counts.sum())

    if requested_total < 0 or requested_total > available_total:
        raise ValueError(
            f"requested_total={requested_total} exceeds "
            f"available_total={available_total}"
        )

    if requested_total == 0:
        return counts * 0

    raw = counts / available_total * requested_total
    allocation = np.floor(raw).astype("int64")
    remaining = int(requested_total - allocation.sum())
    fractional = (raw - allocation).sort_values(ascending=False)

    while remaining > 0:
        progressed = False
        for stratum in fractional.index:
            if allocation.loc[stratum] < counts.loc[stratum]:
                allocation.loc[stratum] += 1
                remaining -= 1
                progressed = True
                if remaining == 0:
                    break
        if not progressed:
            raise RuntimeError("Could not complete proportional allocation.")

    return allocation


def proportional_stratified_sample(
    frame,
    requested_total,
    strata_column,
    random_state,
):

    if requested_total == len(frame):
        return frame.copy()

    strata = (
        frame[strata_column]
        .astype("string")
        .fillna("<missing>")
    )
    counts = strata.value_counts(sort=False)
    allocation = _largest_remainder_allocation(
        counts,
        requested_total,
    )

    sampled_parts = []
    # Stable ordering makes repeated notebook runs deterministic.
    ordered_strata = sorted(allocation.index.astype(str))

    for offset, stratum in enumerate(ordered_strata):
        take = int(allocation.loc[stratum])
        if take == 0:
            continue

        cell = frame.loc[strata.eq(stratum)]
        sampled_parts.append(
            cell.sample(
                n=take,
                replace=False,
                random_state=random_state + offset,
            )
        )

    if not sampled_parts:
        return frame.iloc[0:0].copy()

    return pd.concat(sampled_parts, axis=0)


def match_binary_test_ratio(
    frame,
    target_attack_fraction=TARGET_TEST_ATTACK_FRACTION,
    random_state=RANDOM_STATE,
):

    if not 0.0 < target_attack_fraction < 1.0:
        raise ValueError("target_attack_fraction must be between 0 and 1.")

    working = frame.copy()
    working["__ratio_original_order"] = np.arange(len(working))

    attack_rows = working[
        working[BINARY_COLUMN].eq("attack")
    ]
    benign_rows = working[
        working[BINARY_COLUMN].eq("benign")
    ]

    available_attack = len(attack_rows)
    available_benign = len(benign_rows)

    if available_attack == 0 or available_benign == 0:
        raise ValueError(
            "Every ratio-matched test source must contain both attack "
            "and benign rows."
        )

    target_benign_fraction = 1.0 - target_attack_fraction
    maximum_total = min(
        int(np.floor(available_attack / target_attack_fraction)),
        int(np.floor(available_benign / target_benign_fraction)),
    )

    while maximum_total > 1:
        selected_attack = int(
            round(maximum_total * target_attack_fraction)
        )
        selected_benign = maximum_total - selected_attack

        if (
            0 < selected_attack <= available_attack
            and 0 < selected_benign <= available_benign
        ):
            break

        maximum_total -= 1
    else:
        raise ValueError("Not enough rows to construct a matched test set.")

    sampled_attack = proportional_stratified_sample(
        attack_rows,
        selected_attack,
        LABEL_COLUMN,
        random_state,
    )
    sampled_benign = benign_rows.sample(
        n=selected_benign,
        replace=False,
        random_state=random_state + 10_000,
    )

    matched = (
        pd.concat([sampled_attack, sampled_benign], axis=0)
        .sort_values("__ratio_original_order")
        .drop(columns="__ratio_original_order")
        .reset_index(drop=True)
    )

    actual_attack_fraction = (
        matched[BINARY_COLUMN].eq("attack").mean()
    )

    audit = {
        "available_rows": int(len(frame)),
        "available_attack": int(available_attack),
        "available_benign": int(available_benign),
        "selected_rows": int(len(matched)),
        "selected_attack": int(selected_attack),
        "selected_benign": int(selected_benign),
        "target_attack_pct": 100.0 * target_attack_fraction,
        "actual_attack_pct": 100.0 * actual_attack_fraction,
        "actual_benign_pct": 100.0 * (1.0 - actual_attack_fraction),
    }

    if round(audit["actual_attack_pct"], 1) != round(
        audit["target_attack_pct"], 1
    ):
        raise AssertionError(
            "The matched test set does not round to the requested "
            f"attack prevalence: {audit}"
        )

    return matched, audit


def ratio_match_test_sets(
    test_sets,
    target_attack_fraction=TARGET_TEST_ATTACK_FRACTION,
    random_state=RANDOM_STATE,
):
    matched_sets = {}
    audit_rows = []

    for offset, (test_source, test_frame) in enumerate(
        test_sets.items()
    ):
        matched, audit = match_binary_test_ratio(
            test_frame,
            target_attack_fraction=target_attack_fraction,
            random_state=random_state + 1_000 * offset,
        )
        matched["ratio_match"] = "55.1% attack / 44.9% benign"
        matched_sets[test_source] = matched
        audit_rows.append({
            "test_source": test_source,
            **audit,
        })

    audit_frame = pd.DataFrame(audit_rows)
    return matched_sets, audit_frame


In [ ]:

external_master = df_master.copy()

exp4_external_train_natural, exp4_external_test_natural = (
    split_each_source(
        external_master,
        stratify_column=BINARY_COLUMN,
    )
)


exp4_external_train_balanced, exp4_source_class_quota = (
    balance_binary_external_training(
        exp4_external_train_natural
    )
)

exp4_train_natural = pd.concat(
    [original_train, exp4_external_train_natural],
    ignore_index=True,
)

exp4_train_balanced = pd.concat(
    [original_train, exp4_external_train_balanced],
    ignore_index=True,
)


exp4_natural_test_sets = {
    "original": original_test.copy(),
    **{
        source: part.copy()
        for source, part in exp4_external_test_natural.groupby(
            SOURCE_COLUMN,
            sort=True,
        )
    },
}


exp4_test_sets, exp4_test_ratio_manifest = ratio_match_test_sets(
    exp4_natural_test_sets,
    target_attack_fraction=TARGET_TEST_ATTACK_FRACTION,
    random_state=RANDOM_STATE,
)

print(
    "Balanced external training quota per source/class:",
    exp4_source_class_quota,
)
display(
    source_label_distribution(
        exp4_external_train_balanced,
        BINARY_COLUMN,
    )
)
display(exp4_test_ratio_manifest)


Balanced external training quota per source/class: 9416


,source_pcap,binary_label,count,within_source_pct
0,cic2017,attack,9416,50.0
1,cic2017,benign,9416,50.0
2,cic2018,attack,9416,50.0
3,cic2018,benign,9416,50.0
4,darpa,attack,9416,50.0
5,darpa,benign,9416,50.0
6,unsw-nb15,attack,9416,50.0
7,unsw-nb15,benign,9416,50.0


,test_source,available_rows,available_attack,available_benign,selected_rows,selected_attack,selected_benign,target_attack_pct,actual_attack_pct,actual_benign_pct
0,original,82332,45332,37000,82272,45332,36940,55.1,55.100156,44.899844
1,cic2017,732927,69430,663497,126007,69430,56577,55.1,55.100113,44.899887
2,cic2018,101364,83364,18000,40089,22089,18000,55.1,55.099903,44.900097
3,darpa,864102,28604,835498,51912,28604,23308,55.1,55.100940,44.899060
4,unsw-nb15,100901,4036,96865,7324,4036,3288,55.1,55.106499,44.893501


In [ ]:

split_manifest = pd.DataFrame([
    [
        "exp4_external_train_natural",
        len(exp4_external_train_natural),
    ],
    [
        "exp4_external_train_balanced",
        len(exp4_external_train_balanced),
    ],
    [
        "exp4_combined_train_natural",
        len(exp4_train_natural),
    ],
    [
        "exp4_combined_train_balanced",
        len(exp4_train_balanced),
    ],
    [
        "exp4_external_test_natural",
        len(exp4_external_test_natural),
    ],
    [
        "exp4_ratio_matched_test_rows",
        sum(len(frame) for frame in exp4_test_sets.values()),
    ],
], columns=["dataset", "rows"])

save_frame(
    split_manifest,
    "exp4_ratio55p1_audit_split_manifest.csv",
)
save_frame(
    exp4_test_ratio_manifest,
    "exp4_ratio55p1_test_ratio_manifest.csv",
)


for test_source, test_frame in exp4_test_sets.items():
    save_frame(
        test_frame,
        f"exp4_ratio55p1_test_{test_source}.csv",
    )

display(split_manifest)
display(exp4_test_ratio_manifest)

if GROUP_COLUMN is None:
    print(
        "Record-level external splitting was used because no capture/session "
        "group identifier was configured. Record this as a limitation."
    )


Saved 6 rows -> unsw_nb15_prepared_experiments/exp4_ratio55p1_audit_split_manifest.csv
Saved 5 rows -> unsw_nb15_prepared_experiments/exp4_ratio55p1_test_ratio_manifest.csv
Saved 82,272 rows -> unsw_nb15_prepared_experiments/exp4_ratio55p1_test_original.csv
Saved 126,007 rows -> unsw_nb15_prepared_experiments/exp4_ratio55p1_test_cic2017.csv
Saved 40,089 rows -> unsw_nb15_prepared_experiments/exp4_ratio55p1_test_cic2018.csv
Saved 51,912 rows -> unsw_nb15_prepared_experiments/exp4_ratio55p1_test_darpa.csv
Saved 7,324 rows -> unsw_nb15_prepared_experiments/exp4_ratio55p1_test_unsw-nb15.csv


,dataset,rows
0,exp4_external_train_natural,4198347
1,exp4_external_train_balanced,75328
2,exp4_combined_train_natural,4373688
3,exp4_combined_train_balanced,250669
4,exp4_external_test_natural,1799294
5,exp4_ratio_matched_test_rows,307604


,test_source,available_rows,available_attack,available_benign,selected_rows,selected_attack,selected_benign,target_attack_pct,actual_attack_pct,actual_benign_pct
0,original,82332,45332,37000,82272,45332,36940,55.1,55.100156,44.899844
1,cic2017,732927,69430,663497,126007,69430,56577,55.1,55.100113,44.899887
2,cic2018,101364,83364,18000,40089,22089,18000,55.1,55.099903,44.900097
3,darpa,864102,28604,835498,51912,28604,23308,55.1,55.100940,44.899060
4,unsw-nb15,100901,4036,96865,7324,4036,3288,55.1,55.106499,44.893501


Record-level external splitting was used because no capture/session group identifier was configured. Record this as a limitation.


In [ ]:
MODEL_REGISTRY = {}
MODEL_KEYS = []


def register_binary_model(name, fit_function, predict_function):
    if name in MODEL_REGISTRY:
        raise KeyError(f"Model already registered: {name}")
    MODEL_REGISTRY[name] = {
        "fit": fit_function,
        "predict": predict_function,
    }
    MODEL_KEYS.append(name)
    print(f"Registered binary model: {name}")




In [ ]:

def calculate_binary_metrics(y_true, y_pred, y_probability=None):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    result = {
        "n_test": int(len(y_true)),
        "attack_prevalence": float(np.mean(np.asarray(y_true) == 1)),
        "benign_prevalence": float(np.mean(np.asarray(y_true) == 0)),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "false_positive_rate": fp / (fp + tn) if fp + tn else np.nan,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }
    result["roc_auc"] = (
        roc_auc_score(y_true, y_probability)
        if y_probability is not None and len(np.unique(y_true)) == 2
        else np.nan
    )
    return result


def category_recall_table(frame, y_true, y_pred):
    rows = []
    labels = frame[LABEL_COLUMN].astype(str).to_numpy()
    for category in sorted(pd.unique(labels)):
        mask = labels == category
        yt, yp = y_true[mask], y_pred[mask]
        rows.append({
            "test_category": category,
            "support": int(mask.sum()),
            "attack_recall": (
                recall_score(yt, yp, pos_label=1, zero_division=0)
                if np.any(yt == 1) else np.nan
            ),
            "benign_specificity": (
                recall_score(yt, yp, pos_label=0, zero_division=0)
                if np.any(yt == 0) else np.nan
            ),
        })
    return pd.DataFrame(rows)


In [32]:
def evaluate_model_on_test_sets(
    model_key,
    train_frame,
    test_sets,
    experiment,
    training_variant,
    seed,
):
    adapter = MODEL_REGISTRY[model_key]
    print(
        f"{experiment} | {model_key} | {training_variant} | "
        f"seed={seed} | TRAINING ONCE",
        flush=True,
    )

    started = time.perf_counter()
    fitted = adapter["fit"](train_frame, seed)
    fit_seconds = time.perf_counter() - started

    overall_rows, category_rows, pooled_parts = [], [], []

    for test_source, test_frame in test_sets.items():
        print(f"  Testing {test_source}: {len(test_frame):,} rows", flush=True)
        y_true, y_pred, y_probability, evaluated = adapter["predict"](
            fitted, test_frame
        )
        metrics = calculate_binary_metrics(y_true, y_pred, y_probability)
        overall_rows.append({
            "experiment": experiment,
            "training_variant": training_variant,
            "model": model_key,
            "seed": seed,
            "test_source": test_source,
            "source_role": SOURCE_ROLE.get(test_source, "native_original"),
            "fit_seconds": fit_seconds,
            **metrics,
        })

        categories = category_recall_table(evaluated, y_true, y_pred)
        categories["experiment"] = experiment
        categories["training_variant"] = training_variant
        categories["model"] = model_key
        categories["seed"] = seed
        categories["test_source"] = test_source
        category_rows.append(categories)

        pooled_parts.append(pd.DataFrame({
            "y_true": y_true,
            "y_pred": y_pred,
            "y_probability": y_probability,
            "test_source": test_source,
        }))
        print(
            f"    accuracy={metrics['accuracy']:.4f} "
            f"f1={metrics['f1']:.4f} recall={metrics['recall']:.4f}",
            flush=True,
        )

    pooled = pd.concat(pooled_parts, ignore_index=True)
    pooled_metrics = calculate_binary_metrics(
        pooled["y_true"].to_numpy(),
        pooled["y_pred"].to_numpy(),
        pooled["y_probability"].to_numpy(),
    )
    overall_rows.append({
        "experiment": experiment,
        "training_variant": training_variant,
        "model": model_key,
        "seed": seed,
        "test_source": "POOLED_ALL_ROWS",
        "source_role": "pooled_row_weighted",
        "fit_seconds": fit_seconds,
        **pooled_metrics,
    })

    del fitted
    gc.collect()
    return pd.DataFrame(overall_rows), pd.concat(category_rows, ignore_index=True)


In [33]:
def _merge_save(path, new_data, keys):
    if path.exists():
        old = pd.read_csv(path)
        combined = pd.concat([old, new_data], ignore_index=True)
    else:
        combined = new_data.copy()
    combined = combined.drop_duplicates(subset=keys, keep="last")
    combined.to_csv(path, index=False)
    return combined


def run_experiment(experiment, train_variants, test_sets):
    if not MODEL_KEYS:
        raise RuntimeError(
            "No binary literature models are registered. Add their fit/predict "
            "adapters, then call register_binary_model(...)."
        )

    overall_path = RESULTS_DIR / f"{experiment}_all_runs_partial.csv"
    category_path = RESULTS_DIR / f"{experiment}_category_runs_partial.csv"
    completed = pd.read_csv(overall_path) if overall_path.exists() else pd.DataFrame()
    all_overall = completed.copy()
    all_categories = (
        pd.read_csv(category_path) if category_path.exists() else pd.DataFrame()
    )

    for model_key in MODEL_KEYS:
        for variant, train_frame in train_variants.items():
            for seed in SEEDS:
                expected = set(test_sets) | {"POOLED_ALL_ROWS"}
                if not completed.empty:
                    mask = (
                        completed["model"].eq(model_key)
                        & completed["training_variant"].eq(variant)
                        & completed["seed"].eq(seed)
                    )
                    done = set(completed.loc[mask, "test_source"].astype(str))
                else:
                    done = set()
                if expected.issubset(done):
                    print(f"SKIP completed: {model_key}, {variant}, seed={seed}")
                    continue

                overall, categories = evaluate_model_on_test_sets(
                    model_key, train_frame, test_sets, experiment, variant, seed
                )
                all_overall = _merge_save(
                    overall_path,
                    overall,
                    ["experiment", "training_variant", "model", "seed", "test_source"],
                )
                all_categories = _merge_save(
                    category_path,
                    categories,
                    ["experiment", "training_variant", "model", "seed", "test_source", "test_category"],
                )
                completed = all_overall

    return all_overall, all_categories


def summarize_results(results):
    metrics = [
        "accuracy", "balanced_accuracy", "precision", "recall", "f1",
        "macro_f1", "mcc", "false_positive_rate", "roc_auc",
    ]
    per_source = (
        results.groupby(
            ["experiment", "training_variant", "model", "test_source"],
            dropna=False,
        )[metrics].agg(["mean", "std", "min", "max"]).reset_index()
    )

    non_pooled = results[results["test_source"] != "POOLED_ALL_ROWS"]
    source_macro_seed = (
        non_pooled.groupby(
            ["experiment", "training_variant", "model", "seed"]
        )[metrics].mean().reset_index()
    )
    source_macro = (
        source_macro_seed.groupby(
            ["experiment", "training_variant", "model"]
        )[metrics].agg(["mean", "std"]).reset_index()
    )
    return per_source, source_macro


In [ ]:
# Model execution configuration

from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.impute import SimpleImputer
from scipy.ndimage import gaussian_filter1d

UNSW_CATEGORICAL_COLUMNS = ["proto", "service", "state"]
UNSW_NUMERICAL_COLUMNS = [
    column for column in UNSW_FEATURE_COLUMNS
    if column not in UNSW_CATEGORICAL_COLUMNS
]

MODEL_KEYS = [
    "selected_feature_mlp",
    "vinayakumar_dnn",
    "rf_selected_1dcnn",
    "nb_embedding_svm",       
    "seq2seq_convlstm",      
]

EPOCHS = {
    "vinayakumar_dnn": 20,
    "rf_selected_1dcnn": 50,
    "seq2seq_convlstm": 30,
}

BATCH_SIZES = {
    "vinayakumar_dnn": 64,
    "rf_selected_1dcnn": 64,
    "seq2seq_convlstm": 32,
}

VALIDATION_SIZE = 0.20
MAX_TRAIN_ROWS = None
MAX_TEST_ROWS = None


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


def binary_y(frame):
    if BINARY_COLUMN in frame.columns:
        return frame[BINARY_COLUMN].map(
            {"benign": 0, "attack": 1}
        ).astype("int8").to_numpy()

    if "official_binary_label" in frame.columns:
        return frame["official_binary_label"].astype("int8").to_numpy()

    return np.where(
        frame[LABEL_COLUMN].astype(str).str.casefold().eq("benign"),
        0,
        1,
    ).astype("int8")


def optional_sample(frame, maximum, seed):
    if maximum is None or len(frame) <= maximum:
        return frame.reset_index(drop=True)

    sampled, _ = train_test_split(
        frame,
        train_size=maximum,
        random_state=seed,
        stratify=binary_y(frame),
    )
    return sampled.reset_index(drop=True)


def prepare_test(frame, seed):
    return optional_sample(frame, MAX_TEST_ROWS, seed)


In [ ]:

def fit_ordinal_minmax_preprocessor(train_frame):
    categorical_imputer = SimpleImputer(strategy="most_frequent")
    numerical_imputer = SimpleImputer(strategy="median")

    categorical_values = categorical_imputer.fit_transform(
        train_frame[UNSW_CATEGORICAL_COLUMNS]
    )
    numerical_values = numerical_imputer.fit_transform(
        train_frame[UNSW_NUMERICAL_COLUMNS]
    )

    encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
        dtype=np.float32,
    )
    categorical_encoded = encoder.fit_transform(categorical_values)

    combined = pd.DataFrame(index=train_frame.index)
    for index, column in enumerate(UNSW_CATEGORICAL_COLUMNS):
        combined[column] = categorical_encoded[:, index]
    for index, column in enumerate(UNSW_NUMERICAL_COLUMNS):
        combined[column] = numerical_values[:, index]

    combined = combined[UNSW_FEATURE_COLUMNS]

    scaler = MinMaxScaler(feature_range=(0, 1), clip=True)
    X_train = scaler.fit_transform(combined).astype(np.float32)

    return {
        "categorical_imputer": categorical_imputer,
        "numerical_imputer": numerical_imputer,
        "encoder": encoder,
        "scaler": scaler,
    }, X_train


def transform_ordinal_minmax(preprocessor, frame):
    categorical_values = preprocessor["categorical_imputer"].transform(
        frame[UNSW_CATEGORICAL_COLUMNS]
    )
    numerical_values = preprocessor["numerical_imputer"].transform(
        frame[UNSW_NUMERICAL_COLUMNS]
    )

    categorical_encoded = preprocessor["encoder"].transform(
        categorical_values
    )

    combined = pd.DataFrame(index=frame.index)
    for index, column in enumerate(UNSW_CATEGORICAL_COLUMNS):
        combined[column] = categorical_encoded[:, index]
    for index, column in enumerate(UNSW_NUMERICAL_COLUMNS):
        combined[column] = numerical_values[:, index]

    combined = combined[UNSW_FEATURE_COLUMNS]
    return preprocessor["scaler"].transform(combined).astype(np.float32)


In [36]:
darpa_pcap.columns

Index(['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes',
       'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'sload', 'dload',
       'spkts', 'dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz',
       'dmeansz', 'trans_depth', 'res_bdy_len', 'sjit', 'djit', 'stime',
       'ltime', 'sintpkt', 'dintpkt', 'tcprtt', 'synack', 'ackdat',
       'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login',
       'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm',
       'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'label',
       'source_pcap'],
      dtype='object')

In [ ]:

MLP_SELECTED_FEATURES = [
    "sttl", "ct_srv_dst", "sbytes", "smeansz", "proto",
    "ct_state_ttl", "sloss", "synack", "ct_dst_src_ltm",
    "dmeansz", "ct_srv_src", "service", "ct_dst_sport_ltm",
    "dbytes", "dloss", "state", "tcprtt",
    "ct_src_dport_ltm",
]


def build_selected_feature_gpu_mlp(input_dim):
    """
    TensorFlow/Keras equivalent of the reproduced sklearn MLPClassifier:
    one hidden layer with 150 ReLU units, Adam, initial LR 0.02,
    L2 regularization 0.0001, and binary sigmoid output.
    """
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Dense(
            150,
            activation="relu",
            kernel_regularizer=tf.keras.regularizers.l2(0.0001),
        ),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.02),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=["accuracy"],
    )
    return model


def fit_selected_feature_mlp(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()

    train_frame = optional_sample(train_frame, MAX_TRAIN_ROWS, seed)

    train_part, validation_part = train_test_split(
        train_frame,
        test_size=VALIDATION_SIZE,
        random_state=seed,
        stratify=binary_y(train_frame),
    )

    preprocessor, X_train_all = fit_ordinal_minmax_preprocessor(
        train_part
    )
    X_validation_all = transform_ordinal_minmax(
        preprocessor,
        validation_part,
    )

    selected_indices = [
        UNSW_FEATURE_COLUMNS.index(feature)
        for feature in MLP_SELECTED_FEATURES
    ]

    X_train = X_train_all[:, selected_indices]
    X_validation = X_validation_all[:, selected_indices]

    model = build_selected_feature_gpu_mlp(
        input_dim=len(selected_indices)
    )

    callbacks = [
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=15,
            restore_best_weights=True,
            verbose=1,
        ),
    ]

    print(
        "  Training selected-feature GPU MLP. GPUs:",
        tf.config.list_physical_devices("GPU"),
        flush=True,
    )

    model.fit(
        X_train,
        binary_y(train_part),
        validation_data=(
            X_validation,
            binary_y(validation_part),
        ),
        epochs=20,
        batch_size=256,
        shuffle=True,
        callbacks=callbacks,
        verbose=2,
    )

    return {
        "model": model,
        "preprocessor": preprocessor,
        "selected_indices": selected_indices,
        "seed": seed,
    }


def predict_selected_feature_mlp(fitted, test_frame):
    evaluated = prepare_test(test_frame, fitted["seed"])

    X_test = transform_ordinal_minmax(
        fitted["preprocessor"],
        evaluated,
    )[:, fitted["selected_indices"]]

    probability = fitted["model"].predict(
        X_test,
        batch_size=2048,
        verbose=1,
    ).reshape(-1)

    prediction = (probability >= 0.5).astype("int8")

    return binary_y(evaluated), prediction, probability, evaluated


In [ ]:

def build_vinayakumar_unsw_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(41,)),
        tf.keras.layers.Dense(1024, activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.01),
        tf.keras.layers.Dense(768, activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.01),
        tf.keras.layers.Dense(512, activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.01),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.01),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.01),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.1),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=["accuracy"],
    )
    return model


def fit_vinayakumar_dnn(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()
    train_frame = optional_sample(train_frame, MAX_TRAIN_ROWS, seed)

    train_part, validation_part = train_test_split(
        train_frame,
        test_size=VALIDATION_SIZE,
        random_state=seed,
        stratify=binary_y(train_frame),
    )

    # The reproduced study uses ordinal encoding followed by L2 normalization.
    categorical_encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    )
    X_train_df = train_part[UNSW_FEATURE_COLUMNS].copy()
    X_validation_df = validation_part[UNSW_FEATURE_COLUMNS].copy()

    X_train_df.loc[:, UNSW_CATEGORICAL_COLUMNS] = (
        categorical_encoder.fit_transform(
            X_train_df[UNSW_CATEGORICAL_COLUMNS]
        )
    )
    X_validation_df.loc[:, UNSW_CATEGORICAL_COLUMNS] = (
        categorical_encoder.transform(
            X_validation_df[UNSW_CATEGORICAL_COLUMNS]
        )
    )

    normalizer = Normalizer(norm="l2")
    X_train = normalizer.fit_transform(
        X_train_df.astype(np.float32)
    ).astype(np.float32)
    X_validation = normalizer.transform(
        X_validation_df.astype(np.float32)
    ).astype(np.float32)

    model = build_vinayakumar_unsw_model()

    print("  Training Vinayakumar DNN...", flush=True)
    model.fit(
        X_train,
        binary_y(train_part),
        validation_data=(
            X_validation,
            binary_y(validation_part),
        ),
        epochs=EPOCHS["vinayakumar_dnn"],
        batch_size=BATCH_SIZES["vinayakumar_dnn"],
        shuffle=True,
        verbose=2,
    )

    return {
        "model": model,
        "encoder": categorical_encoder,
        "normalizer": normalizer,
        "seed": seed,
    }


def predict_vinayakumar_dnn(fitted, test_frame):
    evaluated = prepare_test(test_frame, fitted["seed"])
    X_test_df = evaluated[UNSW_FEATURE_COLUMNS].copy()
    X_test_df.loc[:, UNSW_CATEGORICAL_COLUMNS] = (
        fitted["encoder"].transform(
            X_test_df[UNSW_CATEGORICAL_COLUMNS]
        )
    )
    X_test = fitted["normalizer"].transform(
        X_test_df.astype(np.float32)
    ).astype(np.float32)

    probability = fitted["model"].predict(
        X_test,
        batch_size=1024,
        verbose=1,
    ).reshape(-1)
    prediction = (probability >= 0.5).astype("int8")

    return binary_y(evaluated), prediction, probability, evaluated


In [ ]:
NB_SVM_MAX_TEST_ROWS = 20_000
class NaiveBayesFeatureEmbedding:
    def __init__(self, grid_size=2048):
        self.grid_size = grid_size

    def fit(self, X, y):
        self.normal_prior = np.mean(y == 0)
        self.attack_prior = np.mean(y == 1)
        self.log_prior_ratio = np.log(
            self.normal_prior / self.attack_prior
        )

        self.grid_edges = np.linspace(
            -0.10, 1.10, self.grid_size + 1
        )
        self.grid_centres = (
            self.grid_edges[:-1] + self.grid_edges[1:]
        ) / 2
        grid_width = self.grid_edges[1] - self.grid_edges[0]

        self.densities = []
        for feature_index in range(X.shape[1]):
            feature_densities = []
            for target in [0, 1]:
                values = X[y == target, feature_index]
                std = np.std(values, ddof=1)
                iqr = np.percentile(values, 75) - np.percentile(values, 25)
                scales = [
                    value for value in [std, iqr / 1.34]
                    if np.isfinite(value) and value > 0
                ]
                scale = min(scales) if scales else grid_width * 2
                bandwidth = max(
                    0.9 * scale * len(values) ** (-0.2),
                    grid_width * 1.5,
                )

                histogram, _ = np.histogram(
                    values, bins=self.grid_edges
                )
                density = gaussian_filter1d(
                    histogram.astype(float),
                    sigma=bandwidth / grid_width,
                    mode="constant",
                    cval=0.0,
                )
                density = density / (
                    max(density.sum(), 1e-12) * grid_width
                )
                feature_densities.append(
                    np.maximum(density, 1e-12)
                )
            self.densities.append(feature_densities)
        return self

    def transform(self, X):
        embedded = np.empty_like(X, dtype=np.float64)
        for feature_index, (normal_density, attack_density) in enumerate(
            self.densities
        ):
            normal_values = np.interp(
                X[:, feature_index],
                self.grid_centres,
                normal_density,
                left=1e-12,
                right=1e-12,
            )
            attack_values = np.interp(
                X[:, feature_index],
                self.grid_centres,
                attack_density,
                left=1e-12,
                right=1e-12,
            )
            embedded[:, feature_index] = (
                self.log_prior_ratio
                + np.log(normal_values)
                - np.log(attack_values)
            )
        return embedded


def fit_nb_embedding_svm(train_frame, seed):
    set_all_seeds(seed)
    train_frame = optional_sample(train_frame, MAX_TRAIN_ROWS, seed)


    s1, s2 = train_test_split(
        train_frame,
        test_size=0.50,
        random_state=seed,
        stratify=binary_y(train_frame),
    )

    preprocessor, X_s1 = fit_ordinal_minmax_preprocessor(s1)
    X_s2 = transform_ordinal_minmax(preprocessor, s2)

    embedding = NaiveBayesFeatureEmbedding().fit(
        X_s1, binary_y(s1)
    )
    embedded_s2 = embedding.transform(X_s2)

    embedded_scaler = StandardScaler()
    embedded_s2 = embedded_scaler.fit_transform(embedded_s2)

    model = SVC(
        C=1.0,
        kernel="rbf",
        gamma="auto",
        probability=False,
        cache_size=4096,
    )

    print("  Training NB-embedded SVM...", flush=True)
    model.fit(embedded_s2, binary_y(s2))

    return {
        "model": model,
        "preprocessor": preprocessor,
        "embedding": embedding,
        "embedded_scaler": embedded_scaler,
        "seed": seed,
    }


def predict_nb_embedding_svm(fitted, test_frame):
    evaluated = optional_sample(
        test_frame,
        NB_SVM_MAX_TEST_ROWS,
        fitted["seed"],
    )

    print(
        f"  NB-SVM testing on {len(evaluated):,} sampled rows",
        flush=True,
    )

    X_test = transform_ordinal_minmax(
        fitted["preprocessor"],
        evaluated,
    )

    embedded = fitted["embedding"].transform(X_test)
    embedded = fitted["embedded_scaler"].transform(embedded)

    decision = fitted["model"].decision_function(embedded)
    prediction = (decision >= 0).astype("int8")

    return binary_y(evaluated), prediction, decision, evaluated


In [ ]:

def build_rf_cnn():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(15, 1)),
        tf.keras.layers.Conv1D(
            64, 6, padding="same", activation="relu"
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling1D(
            pool_size=3, strides=2, padding="same"
        ),
        tf.keras.layers.Conv1D(
            64, 6, padding="same", activation="relu"
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling1D(
            pool_size=3, strides=2, padding="same"
        ),
        tf.keras.layers.Conv1D(
            64, 6, padding="same", activation="relu"
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling1D(
            pool_size=3, strides=2, padding="same"
        ),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(2, activation="softmax"),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss=tf.keras.losses.CategoricalCrossentropy(),
        metrics=["accuracy"],
    )
    return model


def fit_rf_selected_1dcnn(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()
    train_frame = optional_sample(train_frame, MAX_TRAIN_ROWS, seed)

    train_part, validation_part = train_test_split(
        train_frame,
        test_size=VALIDATION_SIZE,
        random_state=seed,
        stratify=binary_y(train_frame),
    )

    preprocessor, X_train_all = fit_ordinal_minmax_preprocessor(
        train_part
    )
    X_validation_all = transform_ordinal_minmax(
        preprocessor, validation_part
    )

    selector = RandomForestClassifier(
        n_estimators=100,
        criterion="gini",
        max_features="sqrt",
        bootstrap=True,
        random_state=seed,
        n_jobs=-1,
    )
    print("  Fitting Random-Forest feature selector...", flush=True)
    selector.fit(X_train_all, binary_y(train_part))

    selected_indices = np.argsort(
        selector.feature_importances_
    )[-15:][::-1]

    X_train = X_train_all[:, selected_indices].reshape(-1, 15, 1)
    X_validation = X_validation_all[:, selected_indices].reshape(
        -1, 15, 1
    )

    model = build_rf_cnn()
    print("  Training RF-selected 1D CNN...", flush=True)
    model.fit(
        X_train,
        tf.keras.utils.to_categorical(
            binary_y(train_part), num_classes=2
        ),
        validation_data=(
            X_validation,
            tf.keras.utils.to_categorical(
                binary_y(validation_part), num_classes=2
            ),
        ),
        epochs=EPOCHS["rf_selected_1dcnn"],
        batch_size=BATCH_SIZES["rf_selected_1dcnn"],
        shuffle=True,
        verbose=2,
    )

    return {
        "model": model,
        "preprocessor": preprocessor,
        "selected_indices": selected_indices,
        "seed": seed,
    }


def predict_rf_selected_1dcnn(fitted, test_frame):
    evaluated = prepare_test(test_frame, fitted["seed"])
    X_test = transform_ordinal_minmax(
        fitted["preprocessor"], evaluated
    )[:, fitted["selected_indices"]].reshape(-1, 15, 1)

    probability_matrix = fitted["model"].predict(
        X_test, batch_size=1024, verbose=1
    )
    prediction = np.argmax(
        probability_matrix, axis=1
    ).astype("int8")

    return (
        binary_y(evaluated),
        prediction,
        probability_matrix[:, 1],
        evaluated,
    )


In [ ]:

SEQUENCE_LENGTH = 8


def build_seq2seq_convlstm(n_features):
    model_input = tf.keras.layers.Input(
        shape=(SEQUENCE_LENGTH, n_features, 1)
    )

    x = tf.keras.layers.ConvLSTM1D(
        128,
        4,
        padding="same",
        return_sequences=True,
    )(model_input)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(
        pool_size=(2, 2),
        padding="same",
    )(x)
    x = tf.keras.layers.Dropout(0.30)(x)

    x = tf.keras.layers.ConvLSTM1D(
        256,
        4,
        padding="same",
        return_sequences=True,
    )(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    output = tf.keras.layers.Dense(
        2,
        activation="softmax",
    )(x)

    model = tf.keras.Model(model_input, output)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001
        ),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model


def make_sequence_dataset(
    X,
    y,
    shuffle,
    seed,
    batch_size,
):
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.int32)

    print(
        f"Sequence matrix: {X.shape[0]:,} rows × "
        f"{X.shape[1]} features",
        flush=True,
    )

    targets = tf.keras.utils.to_categorical(
        y[SEQUENCE_LENGTH - 1:],
        num_classes=2,
    )

    return tf.keras.utils.timeseries_dataset_from_array(
        data=X[..., np.newaxis],
        targets=targets,
        sequence_length=SEQUENCE_LENGTH,
        sequence_stride=1,
        shuffle=shuffle,
        seed=seed,
        batch_size=batch_size,
    ).prefetch(tf.data.AUTOTUNE)


def fit_seq2seq_convlstm(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()

    train_frame = optional_sample(
        train_frame,
        MAX_TRAIN_ROWS,
        seed,
    ).reset_index(drop=True)

    # Preserve row order instead of randomly mixing rows.
    split_index = int(
        len(train_frame) * (1 - VALIDATION_SIZE)
    )

    train_part = train_frame.iloc[
        :split_index
    ].copy()

    validation_part = train_frame.iloc[
        split_index:
    ].copy()

    preprocessor, X_train = (
        fit_ordinal_minmax_preprocessor(train_part)
    )

    X_validation = transform_ordinal_minmax(
        preprocessor,
        validation_part,
    )

    y_train = binary_y(train_part)
    y_validation = binary_y(validation_part)

    train_dataset = make_sequence_dataset(
        X_train,
        y_train,
        shuffle=True,
        seed=seed,
        batch_size=BATCH_SIZES["seq2seq_convlstm"],
    )

    validation_dataset = make_sequence_dataset(
        X_validation,
        y_validation,
        shuffle=False,
        seed=seed,
        batch_size=BATCH_SIZES["seq2seq_convlstm"],
    )

    model = build_seq2seq_convlstm(
        n_features=X_train.shape[1]
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=3,
            min_delta=1e-4,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
            verbose=1,
        ),
    ]

    print(
        f"Training Seq2Seq ConvLSTM with "
        f"{X_train.shape[1]} features...",
        flush=True,
    )

    model.fit(
        train_dataset,
        validation_data=validation_dataset,
        epochs=EPOCHS["seq2seq_convlstm"],
        callbacks=callbacks,
        verbose=2,
    )

    return {
        "model": model,
        "preprocessor": preprocessor,
        "seed": seed,
        "n_features": X_train.shape[1],
    }


def predict_seq2seq_convlstm(
    fitted,
    test_frame,
):
    evaluated_full = prepare_test(
        test_frame,
        fitted["seed"],
    ).reset_index(drop=True)

    X_test = transform_ordinal_minmax(
        fitted["preprocessor"],
        evaluated_full,
    )

    y_test = binary_y(evaluated_full)

    test_dataset = make_sequence_dataset(
        X_test,
        y_test,
        shuffle=False,
        seed=fitted["seed"],
        batch_size=BATCH_SIZES["seq2seq_convlstm"],
    )

    probabilities = fitted["model"].predict(
        test_dataset,
        verbose=1,
    )

    predictions = np.argmax(
        probabilities,
        axis=1,
    ).astype("int8")

    evaluated = evaluated_full.iloc[
        SEQUENCE_LENGTH - 1:
    ].reset_index(drop=True)

    return (
        y_test[SEQUENCE_LENGTH - 1:],
        predictions,
        probabilities[:, 1],
        evaluated,
    )

In [ ]:

MODEL_REGISTRY = {}
ALL_MODEL_KEYS = []


def register_binary_model(name, fit_function, predict_function):
    MODEL_REGISTRY[name] = {
        "fit": fit_function,
        "predict": predict_function,
    }
    ALL_MODEL_KEYS.append(name)


register_binary_model(
    "selected_feature_mlp",
    fit_selected_feature_mlp,
    predict_selected_feature_mlp,
)
register_binary_model(
    "vinayakumar_dnn",
    fit_vinayakumar_dnn,
    predict_vinayakumar_dnn,
)
register_binary_model(
    "nb_embedding_svm",
    fit_nb_embedding_svm,
    predict_nb_embedding_svm,
)
register_binary_model(
    "rf_selected_1dcnn",
    fit_rf_selected_1dcnn,
    predict_rf_selected_1dcnn,
)
register_binary_model(
    "seq2seq_convlstm",
    fit_seq2seq_convlstm,
    predict_seq2seq_convlstm,
)

unknown = sorted(set(MODEL_KEYS) - set(MODEL_REGISTRY))
if unknown:
    raise KeyError(f"Unknown selected models: {unknown}")

print("Registered models:", ALL_MODEL_KEYS)
print("Enabled models:", MODEL_KEYS)


Registered models: ['selected_feature_mlp', 'vinayakumar_dnn', 'nb_embedding_svm', 'rf_selected_1dcnn', 'seq2seq_convlstm']
Enabled models: ['selected_feature_mlp', 'vinayakumar_dnn', 'rf_selected_1dcnn', 'nb_embedding_svm', 'seq2seq_convlstm']


In [ ]:

EXP4_EXPERIMENT = "exp4_mixed_traffic_test_ratio55p1"

# Train each model once per seed on mixed traffic, then test it separately on
# every ratio-matched source. POOLED_ALL_ROWS concatenates those predictions;
# it is the mixed, row-weighted test result and is not a second model fit.
exp4_overall, exp4_categories = run_experiment(
    EXP4_EXPERIMENT,
    {
        "mixed_external_balanced": exp4_train_balanced,
    },
    exp4_test_sets,
)

exp4_per_source, exp4_source_macro = summarize_results(
    exp4_overall
)

display(exp4_per_source)
display(exp4_source_macro)


exp4_mixed_traffic_test_ratio55p1 | selected_feature_mlp | mixed_external_balanced | seed=11 | TRAINING ONCE
  Training selected-feature GPU MLP. GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Epoch 1/20
784/784 - 10s - 13ms/step - accuracy: 0.9248 - loss: 0.2167 - val_accuracy: 0.9394 - val_loss: 0.1842 - learning_rate: 0.0200
Epoch 2/20
784/784 - 2s - 2ms/step - accuracy: 0.9396 - loss: 0.1779 - val_accuracy: 0.9413 - val_loss: 0.1725 - learning_rate: 0.0200
Epoch 3/20
784/784 - 2s - 2ms/step - accuracy: 0.9411 - loss: 0.1688 - val_accuracy: 0.9414 - val_loss: 0.1658 - learning_rate: 0.0200
Epoch 4/20
784/784 - 2s - 2ms/step - accuracy: 0.9421 - loss: 0.1643 - val_accuracy: 0.9427 - val_loss: 0.1594 - learning_rate: 0.0200
Epoch 5/20
784/784 - 2s - 2ms/step - accuracy: 0.9429 - loss: 0.1605 - val_accuracy: 0.9438 - val_loss: 0.1577 - learning_rate: 0.0200
Epoch 6/20
784/784 - 2s - 2ms/step - accuracy: 0.9432 - loss: 0.1584 - val_accuracy: 0.9445 - val_loss: 

experiment         training_variant  \
                                                                 
0   exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
1   exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
2   exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
3   exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
4   exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
5   exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
6   exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
7   exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
8   exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
9   exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
10  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
11  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
12  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
13  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
14  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
15  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
16  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
17  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
18  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
19  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
20  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
21  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
22  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
23  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
24  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
25  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
26  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
27  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
28  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
29  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   

                   model      test_source  accuracy                      \
                                               mean       std       min   
0       nb_embedding_svm  POOLED_ALL_ROWS  0.938748  0.000606  0.937955   
1       nb_embedding_svm          cic2017  0.980060  0.000972  0.978500   
2       nb_embedding_svm          cic2018  0.989100  0.000354  0.988650   
3       nb_embedding_svm            darpa  0.960420  0.002090  0.957400   
4       nb_embedding_svm         original  0.813950  0.001565  0.811900   
5       nb_embedding_svm        unsw-nb15  0.970044  0.000543  0.969279   
6      rf_selected_1dcnn  POOLED_ALL_ROWS  0.919452  0.018216  0.890222   
7      rf_selected_1dcnn          cic2017  0.982485  0.005278  0.976819   
8      rf_selected_1dcnn          cic2018  0.993410  0.001733  0.991818   
9      rf_selected_1dcnn            darpa  0.964636  0.020712  0.938473   
10     rf_selected_1dcnn         original  0.754966  0.051017  0.673461   
11     rf_selected_1dcnn        unsw-nb15  0.957619  0.018014  0.934189   
12  selected_feature_mlp  POOLED_ALL_ROWS  0.924323  0.003861  0.917618   
13  selected_feature_mlp          cic2017  0.961712  0.005244  0.954010   
14  selected_feature_mlp          cic2018  0.988311  0.007011  0.976078   
15  selected_feature_mlp            darpa  0.955301  0.008002  0.944233   
16  selected_feature_mlp         original  0.812036  0.000988  0.811163   
17  selected_feature_mlp        unsw-nb15  0.972583  0.002038  0.969962   
18      seq2seq_convlstm  POOLED_ALL_ROWS  0.966130  0.001672  0.963368   
19      seq2seq_convlstm          cic2017  0.974454  0.000853  0.972976   
20      seq2seq_convlstm          cic2018  0.989956  0.004533  0.982561   
21      seq2seq_convlstm            darpa  0.963495  0.002957  0.960543   
22      seq2seq_convlstm         original  0.957352  0.004633  0.949176   
23      seq2seq_convlstm        unsw-nb

experiment         training_variant  \
                                                                
0  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
1  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
2  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
3  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
4  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   

                  model  accuracy           balanced_accuracy            \
                             mean       std              mean       std   
0      nb_embedding_svm  0.942715  0.000595          0.938541  0.000562   
1     rf_selected_1dcnn  0.930623  0.016853          0.927696  0.016174   
2  selected_feature_mlp  0.937989  0.003505          0.934315  0.002981   
3      seq2seq_convlstm  0.938981  0.002510          0.938295  0.002388   
4       vinayakumar_dnn  0.683452  0.020104          0.676489  0.023686   

  precision              recall  ...        f1            macro_f1            \
       mean       std      mean  ...      mean       std      mean       std   
0  0.933236  0.000275  0.979459  ...  0.952708  0.000515  0.939264  0.000642   
1  0.927636  0.012717  0.956392  ...  0.940532  0.016106  0.928260  0.016373   
2  0.932941  0.001525  0.970325  ...  0.948086  0.003448  0.934698  0.003452   
3  0.945782  0.004365  0.945016  ...  0.945139  0.002774  0.938150  0.002424   
4  0.711790  0.060428  0.744766  ...  0.694956  0.090744  0.655012  0.036552   

        mcc           false_positive_rate             roc_auc            
       mean       std                mean       std      mean       std  
0  0.891634  0.001105            0.102378  0.000503  0.986493  0.000217  
1  0.861414  0.036329            0.101000  0.016076  0.971235  0.014984  
2  0.881627  0.006833            0.101695  0.002367  0.982023  0.001576  
3  0.876957  0.005095            0.068426  0.006614  0.972594  0.003291  
4  0.392824  0.025678            0.391788  0.232508  0.772064  0.037139  

[5 rows x 21 columns]

In [ ]:

exp4_per_source, exp4_source_macro = summarize_results(
    exp4_overall
)

exp4_category_summary = (
    exp4_categories
    .groupby(
        [
            "experiment",
            "training_variant",
            "model",
            "test_source",
            "test_category",
        ],
        dropna=False,
    )
    .agg(
        support=("support", "first"),
        attack_recall_mean=("attack_recall", "mean"),
        attack_recall_std=("attack_recall", "std"),
        benign_specificity_mean=(
            "benign_specificity",
            "mean",
        ),
        benign_specificity_std=(
            "benign_specificity",
            "std",
        ),
    )
    .reset_index()
)

exp4_overall.to_csv(
    RESULTS_DIR
    / f"{EXP4_EXPERIMENT}_all_runs.csv",
    index=False,
)

exp4_categories.to_csv(
    RESULTS_DIR
    / f"{EXP4_EXPERIMENT}_category_runs.csv",
    index=False,
)

exp4_per_source.to_csv(
    RESULTS_DIR
    / f"{EXP4_EXPERIMENT}_per_source_summary.csv",
    index=False,
)

exp4_source_macro.to_csv(
    RESULTS_DIR
    / f"{EXP4_EXPERIMENT}_source_macro_summary.csv",
    index=False,
)

exp4_category_summary.to_csv(
    RESULTS_DIR
    / f"{EXP4_EXPERIMENT}_category_summary.csv",
    index=False,
)

exp4_test_ratio_manifest.to_csv(
    RESULTS_DIR
    / f"{EXP4_EXPERIMENT}_test_ratio_manifest.csv",
    index=False,
)

print(
    "Primary mixed-test row:",
    "test_source == 'POOLED_ALL_ROWS'",
)
print(
    "Equal-source generalization summary:",
    f"{EXP4_EXPERIMENT}_source_macro_summary.csv",
)

display(exp4_test_ratio_manifest)
display(exp4_per_source)
display(exp4_source_macro)
display(exp4_category_summary)


Primary mixed-test row: test_source == 'POOLED_ALL_ROWS'
Equal-source generalization summary: exp4_mixed_traffic_test_ratio55p1_source_macro_summary.csv


,test_source,available_rows,available_attack,available_benign,selected_rows,selected_attack,selected_benign,target_attack_pct,actual_attack_pct,actual_benign_pct
0,original,82332,45332,37000,82272,45332,36940,55.1,55.100156,44.899844
1,cic2017,732927,69430,663497,126007,69430,56577,55.1,55.100113,44.899887
2,cic2018,101364,83364,18000,40089,22089,18000,55.1,55.099903,44.900097
3,darpa,864102,28604,835498,51912,28604,23308,55.1,55.100940,44.899060
4,unsw-nb15,100901,4036,96865,7324,4036,3288,55.1,55.106499,44.893501


experiment         training_variant  \
                                                                 
0   exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
1   exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
2   exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
3   exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
4   exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
5   exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
6   exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
7   exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
8   exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
9   exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
10  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
11  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
12  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
13  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
14  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
15  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
16  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
17  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
18  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
19  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
20  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
21  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
22  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
23  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
24  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
25  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
26  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
27  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
28  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
29  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   

                   model      test_source  accuracy                      \
                                               mean       std       min   
0       nb_embedding_svm  POOLED_ALL_ROWS  0.938748  0.000606  0.937955   
1       nb_embedding_svm          cic2017  0.980060  0.000972  0.978500   
2       nb_embedding_svm          cic2018  0.989100  0.000354  0.988650   
3       nb_embedding_svm            darpa  0.960420  0.002090  0.957400   
4       nb_embedding_svm         original  0.813950  0.001565  0.811900   
5       nb_embedding_svm        unsw-nb15  0.970044  0.000543  0.969279   
6      rf_selected_1dcnn  POOLED_ALL_ROWS  0.919452  0.018216  0.890222   
7      rf_selected_1dcnn          cic2017  0.982485  0.005278  0.976819   
8      rf_selected_1dcnn          cic2018  0.993410  0.001733  0.991818   
9      rf_selected_1dcnn            darpa  0.964636  0.020712  0.938473   
10     rf_selected_1dcnn         original  0.754966  0.051017  0.673461   
11     rf_selected_1dcnn        unsw-nb15  0.957619  0.018014  0.934189   
12  selected_feature_mlp  POOLED_ALL_ROWS  0.924323  0.003861  0.917618   
13  selected_feature_mlp          cic2017  0.961712  0.005244  0.954010   
14  selected_feature_mlp          cic2018  0.988311  0.007011  0.976078   
15  selected_feature_mlp            darpa  0.955301  0.008002  0.944233   
16  selected_feature_mlp         original  0.812036  0.000988  0.811163   
17  selected_feature_mlp        unsw-nb15  0.972583  0.002038  0.969962   
18      seq2seq_convlstm  POOLED_ALL_ROWS  0.966130  0.001672  0.963368   
19      seq2seq_convlstm          cic2017  0.974454  0.000853  0.972976   
20      seq2seq_convlstm          cic2018  0.989956  0.004533  0.982561   
21      seq2seq_convlstm            darpa  0.963495  0.002957  0.960543   
22      seq2seq_convlstm         original  0.957352  0.004633  0.949176   
23      seq2seq_convlstm        unsw-nb

experiment         training_variant  \
                                                                
0  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
1  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
2  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
3  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   
4  exp4_mixed_traffic_test_ratio55p1  mixed_external_balanced   

                  model  accuracy           balanced_accuracy            \
                             mean       std              mean       std   
0      nb_embedding_svm  0.942715  0.000595          0.938541  0.000562   
1     rf_selected_1dcnn  0.930623  0.016853          0.927696  0.016174   
2  selected_feature_mlp  0.937989  0.003505          0.934315  0.002981   
3      seq2seq_convlstm  0.938981  0.002510          0.938295  0.002388   
4       vinayakumar_dnn  0.683452  0.020104          0.676489  0.023686   

  precision              recall  ...        f1            macro_f1            \
       mean       std      mean  ...      mean       std      mean       std   
0  0.933236  0.000275  0.979459  ...  0.952708  0.000515  0.939264  0.000642   
1  0.927636  0.012717  0.956392  ...  0.940532  0.016106  0.928260  0.016373   
2  0.932941  0.001525  0.970325  ...  0.948086  0.003448  0.934698  0.003452   
3  0.945782  0.004365  0.945016  ...  0.945139  0.002774  0.938150  0.002424   
4  0.711790  0.060428  0.744766  ...  0.694956  0.090744  0.655012  0.036552   

        mcc           false_positive_rate             roc_auc            
       mean       std                mean       std      mean       std  
0  0.891634  0.001105            0.102378  0.000503  0.986493  0.000217  
1  0.861414  0.036329            0.101000  0.016076  0.971235  0.014984  
2  0.881627  0.006833            0.101695  0.002367  0.982023  0.001576  
3  0.876957  0.005095            0.068426  0.006614  0.972594  0.003291  
4  0.392824  0.025678            0.391788  0.232508  0.772064  0.037139  

[5 rows x 21 columns]

,experiment,training_variant,model,test_source,test_category,support,attack_recall_mean,attack_recall_std,benign_specificity_mean,benign_specificity_std
0,exp4_mixed_traffic_test_ratio55p1,mixed_external_balanced,nb_embedding_svm,cic2017,benign,8980,NaN,NaN,0.972249,0.001765
1,exp4_mixed_traffic_test_ratio55p1,mixed_external_balanced,nb_embedding_svm,cic2017,dos,10754,0.991218,0.001198,NaN,NaN
2,exp4_mixed_traffic_test_ratio55p1,mixed_external_balanced,nb_embedding_svm,cic2017,heartbleed,15,0.898266,0.084267,NaN,NaN
3,exp4_mixed_traffic_test_ratio55p1,mixed_external_balanced,nb_embedding_svm,cic2017,infiltration,14,0.000000,0.000000,NaN,NaN
4,exp4_mixed_traffic_test_ratio55p1,mixed_external_balanced,nb_embedding_svm,cic2017,web attack - brute force,153,0.803073,0.021617,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
170,exp4_mixed_traffic_test_ratio55p1,mixed_external_balanced,vinayakumar_dnn,unsw-nb15,fuzzers,1253,0.635435,0.333905,NaN,NaN
171,exp4_mixed_traffic_test_ratio55p1,mixed_external_balanced,vinayakumar_dnn,unsw-nb15,generic,230,0.580000,0.371206,NaN,NaN
172,exp4_mixed_traffic_test_ratio55p1,mixed_external_balanced,vinayakumar_dnn,unsw-nb15,reconnaissance,474,0.635865,0.342568,NaN,NaN
173,exp4_mixed_traffic_test_ratio55p1,mixed_external_balanced,vinayakumar_dnn,unsw-nb15,shellcode,41,0.624390,0.362833,NaN,NaN
